# 第4章：图像滤波## 编程实践：手写高斯滤波与双边滤波| 项目 | 说明 ||------|------|| 输入图片 | `lenaface.jpg`（RGB，来自 Hands-on-CV（上海交通大学《动手学习计算机视觉》）参考资料第 3 章） || 手写核心 | 高斯核生成、二维高斯滤波、双边滤波 || 允许调用 | 仅 `cv_imread` / `cv_imwrite` 图像读写 || 对比验证 | 与 OpenCV `cv2.GaussianBlur` / `cv2.bilateralFilter` 做数值误差对比 |

## 一、学习目标### 📖 学完本章，你应能回答这些问题：1. **什么是空间域滤波？** 为什么对每个像素要"拉上它周围的邻居一起算"？2. **高斯滤波器**的公式、核形状、sigma 参数的作用，并能手写实现。3. **双边滤波器**相比高斯滤波多了什么？为什么它能"保边去噪"？4. 理解**边界填充**、**核归一化**、**可分离性**三个工程细节。### 🎯 什么是"滤波"？——零基础秒懂**滤波 = 对每个像素，结合它周围邻域的像素，重新算一个新值。**想象你在看一幅布满"雪花点"的照片。- 如果你用手把照片**轻微揉模糊**，雪花点会变少，但画面细节也糊了——这就是**高斯滤波**（低通）。- 如果你能"聪明地"只在颜色相近的区域里揉，而遇到明显的边界（比如人脸轮廓）就停手——这就是**双边滤波**（保边去噪）。**数学形式（空间域线性滤波 = 二维卷积）：**$$I_{out}[y, x] = \sum_{dy=-k}^{k} \sum_{dx=-k}^{k} \; w[dy+k, dx+k] \times I_{in}[y+dy, x+dx]$$其中 $w$ 就是**卷积核（Kernel / Filter）**——一个小小的奇数尺寸权重矩阵（如 3×3、5×5）。---### 高斯滤波高斯核由**二维高斯函数**生成（"中间大、四周小"的钟形曲面），然后与图像做卷积：$$G(x, y) = \frac{1}{2\pi\sigma^2} \exp\left( -\frac{x^2 + y^2}{2\sigma^2} \right)$$| 参数 | 影响 ||------|------|| $\sigma$（标准差） | 越大 → 钟形越"胖" → 滤波越强、图像越模糊 || 核窗口大小 | 通常取 $\mathrm{size} = \lceil 6\sigma \rceil$ 并保证为奇数（覆盖 ±3σ，权重已接近 0） |> ⚠️ **必须注意**：高斯核的所有权重加起来必须等于 **1**（归一化）。> 如果不归一化，图像的整体亮度会变亮或变暗。高斯滤波是**线性低通滤波**——它不区分"噪声"和"边缘"，对所有高频成分一刀切，所以**边缘也会被模糊**。---### 双边滤波在高斯滤波的"空间距离权重"基础上，**再乘一个"像素值相似度权重"**：$$w(p, q) = \underbrace{\exp\left( -\frac{\|p-q\|^2}{2\sigma_{space}^2} \right)}_{\text{空间权重：远亲不如近邻}}\;\times\;\underbrace{\exp\left( -\frac{\|I(p)-I(q)\|^2}{2\sigma_{range}^2} \right)}_{\text{灰度权重：非我族类其心必异}}$$| 参数 | 影响 ||------|------|| $\sigma_{space}$ | 控制**多远的邻居**会被纳入计算（越大平滑范围越大） || $\sigma_{range}$ | 控制**多大的灰度差**会被认为是"不同区域"（越大保边能力越弱） |**双边滤波的超能力：保边去噪**- 平坦区域：灰度差都很小 → 两个权重都起作用 → 等效于高斯滤波，噪声被平滑；- 强边缘两侧：灰度差很大 → 灰度权重趋近于 0 → 对面的像素不会被拉进来平均 → 边缘被保留。

In [ ]:
import sysfrom pathlib import Path# 向上查找项目根目录（含 utils.py），并加入 sys.pathROOT = Path.cwd().resolve()while not (ROOT / "utils.py").exists():    if ROOT.parent == ROOT:        raise FileNotFoundError("未找到项目根目录 utils.py")    ROOT = ROOT.parentsys.path.insert(0, str(ROOT))import numpy as npimport cv2import matplotlib.pyplot as pltfrom utils import cv_imread, cv_imwrite, set_random_seed, setup_plot_chinese, show_images, compare_results, plot_gaussian_kernelsetup_plot_chinese()set_random_seed(42)print(f"OpenCV 版本: {cv2.__version__}")print(f"当前工作目录: {Path.cwd()}")print("✅ 工具函数导入成功！")

In [ ]:
# ---------- 零基础直观图：不同 sigma 的高斯核长什么样 ----------print("=" * 60)print("🔍 小 sigma（σ=0.8）：核'尖瘦'，只关注非常近的邻居（轻微模糊）")print("=" * 60)plot_gaussian_kernel(sigma=0.8)print("=" * 60)print("🔍 大 sigma（σ=1.5）：核'矮胖'，覆盖范围更广（更强模糊）")print("=" * 60)plot_gaussian_kernel(sigma=1.5)

## 二、手写约束清单- ✅ 允许：`cv_imread` / `cv_imwrite`；Python 循环与算术；`np.zeros` 开辟空间。- ❌ 禁止：`cv2.GaussianBlur` / `cv2.bilateralFilter` / `cv2.filter2D` 用于实现（仅可用于对比验证）。- ✅ 可视化：`matplotlib` 仅用于显示。- 边界：本页统一使用**零填充（zero padding）**，对比验证时只比较图像内部区域。

In [ ]:
def gaussian_kernel_manual(size, sigma):    """手写二维高斯核。size 为奇数边长，sigma 为标准差。"""    kernel = np.zeros((size, size), dtype=np.float64)    center = size // 2    s2 = 2.0 * sigma * sigma    total = 0.0    for i in range(size):        for j in range(size):            x = i - center            y = j - center            kernel[i, j] = np.exp(-(x * x + y * y) / s2) / (np.pi * s2)            total += kernel[i, j]    return kernel / totaldef gaussian_blur_manual(image, sigma, kernel_size=5):    """手写二维高斯滤波（零填充，支持灰度与彩色图）。"""    kernel = gaussian_kernel_manual(kernel_size, sigma)    pad = kernel_size // 2    if image.ndim == 2:        h, w = image.shape        padded = np.zeros((h + 2 * pad, w + 2 * pad), dtype=np.float64)        padded[pad:pad + h, pad:pad + w] = image.astype(np.float64)        out = np.zeros((h, w), dtype=np.float64)        for y in range(h):            for x in range(w):                acc = 0.0                for ky in range(kernel_size):                    for kx in range(kernel_size):                        acc += padded[y + ky, x + kx] * kernel[ky, kx]                out[y, x] = acc        return np.clip(out, 0, 255).astype(np.uint8)    h, w, c = image.shape    padded = np.zeros((h + 2 * pad, w + 2 * pad, c), dtype=np.float64)    padded[pad:pad + h, pad:pad + w, :] = image.astype(np.float64)    out = np.zeros((h, w, c), dtype=np.float64)    for y in range(h):        for x in range(w):            for ch in range(c):                acc = 0.0                for ky in range(kernel_size):                    for kx in range(kernel_size):                        acc += padded[y + ky, x + kx, ch] * kernel[ky, kx]                out[y, x, ch] = acc    return np.clip(out, 0, 255).astype(np.uint8)def bilateral_filter_manual(image, sigma_space, sigma_range, kernel_size=5):    """手写双边滤波（零填充）。"""    pad = kernel_size // 2    s2 = 2.0 * sigma_space * sigma_space    r2 = 2.0 * sigma_range * sigma_range    center = kernel_size // 2    if image.ndim == 2:        h, w = image.shape        padded = np.zeros((h + 2 * pad, w + 2 * pad), dtype=np.float64)        padded[pad:pad + h, pad:pad + w] = image.astype(np.float64)        out = np.zeros((h, w), dtype=np.float64)        for y in range(h):            for x in range(w):                norm = 0.0                acc = 0.0                for ky in range(kernel_size):                    for kx in range(kernel_size):                        dy, dx = ky - center, kx - center                        spatial = np.exp(-(dy * dy + dx * dx) / s2)                        range_w = np.exp(-((padded[y + ky, x + kx] - padded[y + center, x + center]) ** 2) / r2)                        weight = spatial * range_w                        acc += padded[y + ky, x + kx] * weight                        norm += weight                out[y, x] = acc / (norm + 1e-9)        return np.clip(out, 0, 255).astype(np.uint8)    h, w, c = image.shape    padded = np.zeros((h + 2 * pad, w + 2 * pad, c), dtype=np.float64)    padded[pad:pad + h, pad:pad + w, :] = image.astype(np.float64)    out = np.zeros((h, w, c), dtype=np.float64)    for y in range(h):        for x in range(w):            for ch in range(c):                norm = 0.0                acc = 0.0                for ky in range(kernel_size):                    for kx in range(kernel_size):                        dy, dx = ky - center, kx - center                        spatial = np.exp(-(dy * dy + dx * dx) / s2)                        range_w = np.exp(-((padded[y + ky, x + kx, ch] - padded[y + center, x + center, ch]) ** 2) / r2)                        weight = spatial * range_w                        acc += padded[y + ky, x + kx, ch] * weight                        norm += weight                out[y, x, ch] = acc / (norm + 1e-9)    return np.clip(out, 0, 255).astype(np.uint8)def compare_interior(manual, reference, border, name):    """只比较去掉 border 边界的内部区域。"""    a = manual.astype(np.float64)    b = reference.astype(np.float64)    if a.ndim == 2:        a, b = a[border:-border, border:-border], b[border:-border, border:-border]    else:        a, b = a[border:-border, border:-border, :], b[border:-border, border:-border, :]    diff = np.abs(a - b)    print(f"[{name}] 内部区域对比: MAE={diff.mean():.6f}, RMSE={np.sqrt((diff**2).mean()):.6f}, Max={diff.max():.6f}")

In [ ]:
# 读取彩色图像并手写高斯滤波img = cv_imread("lenaface.jpg", cv2.IMREAD_COLOR)assert img is not None, "读取 lenaface.jpg 失败"sigma, ksize = 1.0, 5gaussian = gaussian_blur_manual(img, sigma, ksize)# ---------- 与 OpenCV 对比验证 ----------gaussian_cv = cv2.GaussianBlur(img, (ksize, ksize), sigmaX=sigma)compare_interior(gaussian, gaussian_cv, border=ksize // 2, name="高斯滤波")cv_imwrite("gaussian_filtered.jpg", gaussian)show_images([img, gaussian], ["原图", f"高斯滤波 sigma={sigma}"], figsize=(9, 4))

In [ ]:
# 手写双边滤波sigma_space, sigma_range, ksize = 20.0, 40.0, 5bilateral = bilateral_filter_manual(img, sigma_space, sigma_range, ksize)# ---------- 与 OpenCV 对比验证 ----------bilateral_cv = cv2.bilateralFilter(img, ksize, sigma_range, sigma_space)compare_interior(bilateral, bilateral_cv, border=ksize // 2, name="双边滤波")cv_imwrite("bilateral_filtered.jpg", bilateral)show_images([img, gaussian, bilateral], ["原图", "高斯滤波", "双边滤波(保边)"], figsize=(13, 4))

## 三、结果与参数分析- 高斯滤波对整幅图无差别平滑，边缘也会模糊。- 双边滤波在平坦区域平滑噪声、在强边缘处保留边缘，适合去噪而不糊边。- 增大 `sigma_space` 会让空间平滑范围变大；增大 `sigma_range` 会让更多灰度差被当作"平坦"，保边能力下降。**易错点**1. 高斯核必须**归一化**（除以权重总和），否则图像会整体变亮/变暗。2. 边界填充方式不同会导致边缘像素与 OpenCV 有差异；本页因此只比较内部区域。3. 双边滤波的灰度权重要用**中心像素**的灰度与邻域灰度比较，不能复用像素绝对坐标。

## 四、科研规范小结1. **核生成与卷积分离**：`gaussian_kernel_manual` 可独立测试（如核之和是否等于 1）。2. **边界处理透明**：明确写出 zero padding，并用 `compare_interior` 排除边界歧义。3. **定量对比**：MAE/RMSE/Max 同时报告，比单纯目测更严谨。

## 五、练习：均值滤波与中值滤波去椒盐噪声**要求**：手写 `mean_filter_manual`、`median_filter_manual`、`add_salt_pepper_noise`，对加噪图像分别做均值/中值滤波，并对比去噪效果。

In [ ]:
# ==================== 练习解决方案 ====================import randomdef add_salt_pepper_noise(image, ratio=0.05):    """手写椒盐噪声：随机把像素置为 255 或 0。"""    result = image.copy()    h, w = image.shape[:2]    n = int(h * w * ratio)    for _ in range(n):        y = random.randint(0, h - 1)        x = random.randint(0, w - 1)        if image.ndim == 2:            result[y, x] = 255 if random.random() > 0.5 else 0        else:            v = 255 if random.random() > 0.5 else 0            result[y, x, :] = v    return resultdef mean_filter_manual(image, kernel_size=3):    """手写均值滤波。"""    pad = kernel_size // 2    if image.ndim == 2:        h, w = image.shape        out = np.zeros((h, w), dtype=np.uint8)        padded = np.pad(image.astype(np.float64), pad, mode="reflect")        for y in range(h):            for x in range(w):                s = 0.0                for ky in range(kernel_size):                    for kx in range(kernel_size):                        s += padded[y + ky, x + kx]                out[y, x] = int(round(s / (kernel_size * kernel_size)))        return out    h, w, c = image.shape    out = np.zeros((h, w, c), dtype=np.uint8)    for ch in range(c):        out[:, :, ch] = mean_filter_manual(image[:, :, ch], kernel_size)    return outdef median_filter_manual(image, kernel_size=3):    """手写中值滤波：对窗口内像素排序后取中值。"""    pad = kernel_size // 2    if image.ndim == 2:        h, w = image.shape        padded = np.pad(image.astype(np.uint8), pad, mode="reflect")        out = np.zeros((h, w), dtype=np.uint8)        win = kernel_size * kernel_size        for y in range(h):            for x in range(w):                vals = []                for ky in range(kernel_size):                    for kx in range(kernel_size):                        vals.append(int(padded[y + ky, x + kx]))                vals.sort()                out[y, x] = vals[win // 2]        return out    h, w, c = image.shape    out = np.zeros((h, w, c), dtype=np.uint8)    for ch in range(c):        out[:, :, ch] = median_filter_manual(image[:, :, ch], kernel_size)    return outnoisy = add_salt_pepper_noise(img, 0.05)mean_m = mean_filter_manual(noisy, 3)median_m = median_filter_manual(noisy, 3)mean_cv = cv2.blur(noisy, (3, 3))median_cv = cv2.medianBlur(noisy, 3)compare_results(mean_m, mean_cv, "均值滤波")compare_results(median_m, median_cv, "中值滤波")cv_imwrite("noisy.jpg", noisy)cv_imwrite("mean_denoised.jpg", mean_m)cv_imwrite("median_denoised.jpg", median_m)show_images([noisy, mean_m, median_m], ["椒盐噪声", "均值滤波", "中值滤波"], figsize=(13, 4))

### 代码要点解释1. **椒盐噪声**：随机选择像素置为极值，用来模拟传感器坏点/传输错误。2. **均值滤波**：对噪声也取平均，因此去噪后仍会留下灰斑。3. **中值滤波**：排序取中值，极值点（0/255）不会被选中，对椒盐噪声更有效。